# Rare Variant Association Pipeline
Pipeline steps:
1. VCF annotation with snpEff + ANNOVAR
2. Variant filtering (exonic/splicing, non-synonymous)
3. VCF extraction + PLINK file creation
4. Association tests: PLINK assoc/fisher/GLM + RVtests SKAT/SKAT-O
5. Result aggregation into Excel workbook

## Imports

In [ ]:
import os
import subprocess
import re
import gzip
import pandas as pd
import polars as pl
import numpy as np
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

In [ ]:
HESTIA         = f"/path/to/home" 
VCF_DIR        = f"{HOME}/INPUT/VCF"          # directory containing chr{N}.vcf.gz
GENE_LIST_BED  = f"{HOME}input/analysis/PD_Genes_n44_NCBI_RefSeq_Select_and_MANE_hg38.bed"     # UCSC BED: chr  start  end  gene  (chr-prefixed)
COVARIATE_BASE = f"{HOME}input/meta/CATPD_covariate_for_plink"       # base name – expects .pheno and .cov files
KEEP_FILE      = f'{HOME}input/meta/CATPD_covariate_for_plink.keep'
OUT_DIR        = f"{HOME}input/analysis/output"           # all outputs go here
WD             = OUT_DIR    # working dir used by PLINK helpers
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

In [ ]:
TOOL_DIR       = f"/path/to/tools/"
SNPEFF_JAR     = os.path.expanduser("~/snpEff/snpEff.jar")
SNPEFF_DB      = "GRCh38.115"
ANNOVAR_PL     = f"{TOOL_DIR}/annovar_2025/table_annovar.pl"
ANNOVAR_DB     = f"{TOOL_DIR}/annovar_2025/humandb"
RVTEST_BIN     = f"{TOOL_DIR}/rvtests/executable/rvtest"
REF_FLAT       = f"{TOOL_DIR}/annovar_2025/refFlat.txt"

Modify the following parameters to change MAF and MAC thresholds

In [ ]:
CHROMOSOMES    = list(range(1, 23))
THREADS        = 16
MAC_LOW        = 1
MAX_MAF        = 0.05
CI             = 0.95

In [ ]:
# Covariates for all tests
PLINK_COVARS   = ["SEX", "AGE"] + [f"PC{i}" for i in range(1, 11)]
RVTEST_COVARS  = "SEX,AGE,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10"
PHENO_NAME     = "DISEASE"

In [ ]:
# GLM output columns
GLM_COLS = [
    "+chrom","+pos","+ref","+alt", "omitted","+nobs"
    "+p", "+orbeta", "+se", "+a1freqcc","+a1countcc","+a1count",
    "+totallele","+totallelecc","+gcountcc","+ci"]

## Per-chromosome: snpEff + ANNOVAR annotation

In [ ]:
def run(cmd, **kwargs):
    print("CMD:", " ".join(str(c) for c in cmd))
    result = subprocess.run([str(c) for c in cmd], check=True, **kwargs)
    return result

In [ ]:
def annotate_chromosome(chnum):

    chr_tag   = f"chr{chnum}"
    vcf_in    = os.path.join(VCF_DIR, f"{chr_tag}.vcf.gz")
    chrom_dir = os.path.join(OUT_DIR, chr_tag)
    Path(chrom_dir).mkdir(exist_ok=True)

    #1a. Build per-chromosome region file from gene list BED 
    region_file = os.path.join(chrom_dir, f"{chr_tag}_regions.bed")
    bed = pd.read_csv(GENE_LIST_BED, sep="\t", header=None,
                      names=["chrom","start","end","gene"])
    chr_bed = bed[bed["chrom"] == chr_tag].copy()
    chr_bed[["chrom","start","end"]].to_csv(region_file, sep="\t",
                                             header=False, index=False)
    chr_genes = set(chr_bed["gene"].unique())
    print(f"[{chr_tag}] {len(chr_genes)} genes in region file.")

    if chr_bed.empty:
        print(f"[{chr_tag}] No genes – skipping.")
        return None, None, None

    #1b. bcftools + snpEff -
    snpeff_vcf = os.path.join(chrom_dir, f"{chr_tag}.snpeff.vcf")
    bcftools_cmd = ["bcftools", "view", "-G", "-R", region_file, vcf_in]
    snpeff_cmd  = ["java", "-jar", SNPEFF_JAR, SNPEFF_DB]

    with open(snpeff_vcf, "w") as fout:
        p1 = subprocess.Popen(bcftools_cmd, stdout=subprocess.PIPE)
        p2 = subprocess.Popen(snpeff_cmd,  stdin=p1.stdout,
                               stdout=fout, stderr=subprocess.PIPE)
        p1.stdout.close()
        _, err = p2.communicate()
        if p2.returncode != 0:
            print(f"[{chr_tag}] snpEff stderr:", err.decode()[:500])
    print(f"[{chr_tag}] snpEff done → {snpeff_vcf}")

    #1c. ANNOVAR broad annotation (for filtering) 
    annovar_prefix = os.path.join(chrom_dir, f"{chr_tag}_annovar_filter")
    run(["perl", ANNOVAR_PL, snpeff_vcf, ANNOVAR_DB,
         "-buildver", "hg38",
         "-out", annovar_prefix,
         "-remove",
         "-protocol", "refGene",
         "-operation", "g",
         "-nastring", ".",
         "-vcfinput",
         "thread", "1"])

    annovar_txt = f"{annovar_prefix}.hg38_multianno.txt"
    print(f"[{chr_tag}] ANNOVAR broad done → {annovar_txt}")
    return snpeff_vcf, annovar_txt, chr_genes

In [ ]:
#Step 0: ensure every input VCF is indexed (parallel) 
INDEX_WORKERS = min(8, len(CHROMOSOMES))  # tune to available cores

def ensure_index(chnum):
    chr_tag = f"chr{chnum}"
    vcf     = os.path.join(VCF_DIR, f"{chr_tag}.vcf.gz")
    if not os.path.exists(vcf):
        return chnum, "missing"
    tbi = vcf + ".tbi"
    csi = vcf + ".csi"
    if os.path.exists(tbi) or os.path.exists(csi):
        return chnum, "ok"
    # Index is missing – build it
    print(f"[{chr_tag}] index missing – running bcftools index ...")
    subprocess.run(["bcftools", "index", "tbi", "threads", "2", vcf],
                   check=True, capture_output=True)
    return chnum, "indexed"

print(f"Checking / building VCF indices with {INDEX_WORKERS} parallel workers ...")

missing_vcfs = []
with ThreadPoolExecutor(max_workers=INDEX_WORKERS) as pool:
    futures = {pool.submit(ensure_index, c): c for c in CHROMOSOMES}
    for fut in as_completed(futures):
        chnum, status = fut.result()
        if status == "missing":
            print(f"  [chr{chnum}] VCF not found – will skip")
            missing_vcfs.append(chnum)
        elif status == "indexed":
            print(f"  [chr{chnum}] index built")

CHROMOSOMES_TO_RUN = [c for c in CHROMOSOMES if c not in missing_vcfs]
print(f"\nReady to annotate {len(CHROMOSOMES_TO_RUN)} chromosomes.")

#Step 1: run annotation for all chromosomes 
chrom_results = {} 

for chnum in CHROMOSOMES_TO_RUN:
    try:
        sv, at, cg = annotate_chromosome(chnum)
        chrom_results[chnum] = (sv, at, cg)
    except Exception as e:
        print(f"[chr{chnum}] ERROR: {e}")

## Variant filtering

Keep variants where:
- `Gene.refGene` is in the gene list for that chromosome
- `Func.refGene` is `exonic` or `splicing`
- `ExonicFunc.refGene` is **not** `synonymous SNV`

In [ ]:
def filter_variants(annovar_txt, chr_genes):

    df = pd.read_csv(annovar_txt, sep="\t", low_memory=False)
    df.columns = [c.strip() for c in df.columns]
    gene_col   = next(c for c in df.columns if "Gene.refGene"       in c)
    func_col   = next(c for c in df.columns if "Func.refGene"       in c)
    exfunc_col = next(c for c in df.columns if "ExonicFunc.refGene" in c)

    print(f"  Total variants in file : {len(df)}")
    print(f"  Unique Func values     : {df[func_col].unique()}")
    print(f"  Unique ExonicFunc values: {df[exfunc_col].unique()}")

    def row_passes(row):
        sep = re.compile(r"[;,]")
        genes   = [g.strip() for g in sep.split(str(row[gene_col]))]
        funcs   = [f.strip() for f in sep.split(str(row[func_col]))]
        exfuncs = [e.strip() for e in sep.split(str(row[exfunc_col]))]

        # Pad shorter lists with their last value so indices align
        n = len(genes)
        funcs   = (funcs   + [funcs[-1]]   * n)[:n]
        exfuncs = (exfuncs + [exfuncs[-1]] * n)[:n]

        for gene, func, exfunc in zip(genes, funcs, exfuncs):
            if gene not in chr_genes:
                continue
            if not re.search(r"exonic|splicing", func, re.IGNORECASE):
                continue
            if exfunc == "synonymous SNV":
                continue
            return True
        return False

    mask = df.apply(row_passes, axis=1)
    passed = df[mask].copy()
    print(f"Final passing variants: {len(passed)} / {len(df)}")
    return passed


# Build combined pass-filter table
all_passed = {}
for chnum, (sv, at, cg) in chrom_results.items():
    if at is None:
        continue
    print(f"chr{chnum}:")
    try:
        all_passed[chnum] = filter_variants(at, cg)
    except Exception as e:
        print(f"  ERROR: {e}")

## Extract filtered variants from original VCF

In [ ]:
def variants_to_region_file(passed_df, region_path):
    reg = passed_df[["Chr", "Start"]].copy()
    reg.columns = ["CHROM", "POS"]
    reg = reg.drop_duplicates()
    reg.to_csv(region_path, sep="\t", header=False, index=False)
    return region_path


def extract_genotypes(chnum, passed_df):
    chr_tag   = f"chr{chnum}"
    vcf_in    = os.path.join(VCF_DIR, f"{chr_tag}.vcf.gz")
    chrom_dir = os.path.join(OUT_DIR, chr_tag)
    out_base  = os.path.join(chrom_dir, f"{chr_tag}_filtered")

    region_path = out_base + "_regions.txt"
    variants_to_region_file(passed_df, region_path)

    # Extract with genotypes
    out_vcf    = out_base + ".vcf"
    out_vcf_gz = out_base + ".vcf.gz"
    run(["bcftools", "view", "-R", region_path, vcf_in, "-o", out_vcf])
    run(["bgzip", "-f", out_vcf])
    run(["tabix", "-p", "vcf", out_vcf_gz])
    print(f"[{chr_tag}] VCF extracted → {out_vcf_gz}")

    # PLINK2 (pfiles)
    plink2_prefix = out_base + "_plink2"
    run(["plink2",
         "vcf", out_vcf_gz,
         "vcf-half-call", "m",
         "make-pgen",
         "out", plink2_prefix,
         "threads", THREADS])

    # PLINK 1.9 (bfiles)
    plink1_prefix = out_base + "_plink1"
    run(["plink2",
         "pfile", plink2_prefix,
         "make-bed",
         "out", plink1_prefix,
         "threads", THREADS])

    return out_vcf_gz, plink2_prefix, plink1_prefix


# Run extraction
extraction_results = {}
for chnum, passed_df in all_passed.items():
    if passed_df.empty:
        print(f"chr{chnum}: no variants to extract.")
        continue
    try:
        extraction_results[chnum] = extract_genotypes(chnum, passed_df)
    except Exception as e:
        print(f"chr{chnum} extraction ERROR: {e}")

## Association testing
### PLINK assoc / fisher / GLM


In [ ]:
def assoc(inputFile, outPrefix, rangeFile, threads, fr_up, mac_low,
          covariate, plink2_covars, ci=0.95):
    glmCols = ["+orbeta","+a1freqcc","+a1countcc","+a1count",
               "+totallele","+totallelecc","+gcountcc","+ci"]
    keep_file = KEEP_FILE

    #plink 1.9 keep file: prepend FID=0 column, no header 
    keep19_file = outPrefix + "_plink19.keep"
    with open(keep_file) as fin, open(keep19_file, "w") as fout:
        for line in fin:
            line = line.rstrip("\n")
            if not line:
                continue
            parts = line.split("\t")
            iid = parts[1] if len(parts) >= 2 else parts[0]
            fout.write(f"0\t{iid}\n")

    #plink 1.9 pheno file: prepend 0 column, drop header 
    pheno19_file = outPrefix + "_plink19.pheno"
    pheno_df = pd.read_csv(f"{covariate}.pheno", sep=r"\s+", engine="python")
    pheno_df.insert(0, "FID", 0)
    pheno_df.to_csv(pheno19_file, sep="\t", header=False, index=False)

    #plink 1.9 covar file: prepend 0 column, drop header 
    cov19_file = outPrefix + "_plink19.cov"
    cov_df = pd.read_csv(f"{covariate}.cov", sep=r"\s+", engine="python")
    cov_df.insert(0, "FID", 0)
    cov_df.to_csv(cov19_file, sep="\t", header=False, index=False)

    try:
        # Step 1: extract variant subset → pgen
        run(["plink2",
             "pfile", inputFile,
             "set-all-var-ids", "@:#:$r:$a",
             "keep", keep_file,
             "extract", "range", rangeFile,
             "mac", str(mac_low),
             "max-maf", str(fr_up),
             "threads", str(threads),
             "make-pgen",
             "out", outPrefix])

        # Step 2: convert to bfile
        run(["plink2",
             "pfile", outPrefix,
             "keep", keep_file,
             "threads", str(threads),
             "make-bed",
             "out", outPrefix])

        #Chi-square (plink 1.9) 
        run(["plink",
             "bfile", outPrefix,
             "keep", keep19_file,
             "pheno", pheno19_file,
             "assoc", "adjust",
             "keep-allele-order", "allow-no-sex",
             "mac", str(mac_low), "max-maf", str(fr_up),
             "ci", str(ci),
             "threads", str(threads),
             "make-bed",
             "out", f"{outPrefix}.chi"])

        #Fisher (plink 1.9) 
        run(["plink",
             "bfile", outPrefix,
             "keep", keep19_file,
             "pheno", pheno19_file,
             "fisher", "adjust",
             "keep-allele-order", "allow-no-sex",
             "mac", str(mac_low), "max-maf", str(fr_up),
             "ci", str(ci),
             "threads", str(threads),
             "make-bed",
             "out", f"{outPrefix}.fisher"])

        #Logistic – no covariates (plink2) 
        run(["plink2",
             "pfile", outPrefix,
             "keep", keep_file,
             "glm", "allow-no-covars", 
             f"cols={','.join(glmCols)}",
             "adjust",
             "mac", str(mac_low), "max-maf", str(fr_up),
             "ci", str(ci),
             "pheno", f"{covariate}.pheno",
             "threads", str(threads),
             "make-bed",
             "out", f"{outPrefix}.nocovar"])

        #Logistic – with covariates (plink2) 
        run(["plink2",
             "pfile", outPrefix,
             "keep", keep_file,
             "glm", "hide-covar",
             f"cols={','.join(glmCols)}",
             "adjust",
             "mac", str(mac_low), "max-maf", str(fr_up),
             "pheno", f"{covariate}.pheno",
             "covar", f"{covariate}.cov",
             "covar-name", ",".join(plink2_covars),
             "covar-variance-standardize",
             "ci", str(ci),
             "threads", str(threads),
             "make-bed",
             "out", f"{outPrefix}.covar"])

    except subprocess.CalledProcessError as e:
        print(f"PLINK error at {outPrefix}: {e}")


print("assoc() defined.")


In [ ]:
def strip_chr_prefix_plink(plink2_prefix, plink1_prefix, chrom_dir, chr_tag):
    import shutil

    def strip_chrom_col(src, dst, sep="\t", chrom_col=0, comment="#"):
        with open(src) as fin, open(dst, "w") as fout:
            for line in fin:
                if line.startswith(comment):
                    fout.write(line)
                    continue
                parts = line.rstrip("\n").split(sep)
                parts[chrom_col] = re.sub(r"^chr", "", parts[chrom_col])
                fout.write(sep.join(parts) + "\n")

    p2_nochr = plink2_prefix + "_nochr"
    p1_nochr = plink1_prefix + "_nochr"

    #plink2 pfiles: copy .psam unchanged, rewrite .pvar CHROM 
    for ext in [".psam", ".pgen"]:
        src = plink2_prefix + ext
        if os.path.exists(src):
            shutil.copy2(src, p2_nochr + ext)
    pvar_src = plink2_prefix + ".pvar"
    pvar_dst = p2_nochr      + ".pvar"
    if os.path.exists(pvar_src):
        strip_chrom_col(pvar_src, pvar_dst, sep="\t", chrom_col=0, comment="#")

    #plink1 bfiles: copy .bed/.fam unchanged, rewrite .bim CHROM 
    for ext in [".bed", ".fam"]:
        src = plink1_prefix + ext
        if os.path.exists(src):
            shutil.copy2(src, p1_nochr + ext)
    bim_src = plink1_prefix + ".bim"
    bim_dst = p1_nochr      + ".bim"
    if os.path.exists(bim_src):
        # .bim is space-delimited; CHROM is column 0
        strip_chrom_col(bim_src, bim_dst, sep="\t", chrom_col=0, comment="")

    print(f"  [{chr_tag}] nochr pfiles → {p2_nochr}")
    print(f"  [{chr_tag}] nochr bfiles → {p1_nochr}")
    return p2_nochr, p1_nochr


# Run PLINK association tests for each chromosome
for chnum, (vcf_gz, plink2_prefix, plink1_prefix) in extraction_results.items():
    chr_tag   = f"chr{chnum}"
    chrom_dir = os.path.join(OUT_DIR, chr_tag)

    #Build range file using numeric chromosome (no 'chr' prefix) 
    # PLINK extract range expects the chromosome format to match the .pvar/.bim,
    # which is numeric-only (e.g. '1', not 'chr1').
    range_file = os.path.join(chrom_dir, f"{chr_tag}_ranges_nochr.bed")
    bed_df = pd.read_csv(GENE_LIST_BED, sep="\t", header=None,
                         names=["chrom","start","end","gene"])
    chr_bed = bed_df[bed_df["chrom"] == chr_tag][["chrom","start","end"]].copy()
    chr_bed["chrom"] = chr_bed["chrom"].str.replace(r"^chr", "", regex=True)
    chr_bed.to_csv(range_file, sep="\t", header=False, index=False)

    #Rewrite pvar/bim to strip 'chr' prefix → new _nochr file set 
    p2_nochr, p1_nochr = strip_chr_prefix_plink(
        plink2_prefix, plink1_prefix, chrom_dir, chr_tag
    )

    out_base = os.path.join(chrom_dir, f"{chr_tag}_assoc")
    print(f"\n chr{chnum}: running PLINK tests ")
    assoc(inputFile=p2_nochr,
          outPrefix=out_base,
          rangeFile=range_file,
          threads=THREADS,
          fr_up=MAX_MAF,
          mac_low=MAC_LOW,
          covariate=COVARIATE_BASE,
          plink2_covars=PLINK_COVARS,
          ci=CI)


### RVtests SKAT / SKAT-O

In [ ]:
def make_rvtests_files(covariate, out_base):
    rv_file = out_base + ".rvt.txt"

    pheno_df = pd.read_csv(f"{covariate}.pheno", sep=r"\s+", engine="python")
    cov_df   = pd.read_csv(f"{covariate}.cov",   sep=r"\s+", engine="python")

    # Normalise: strip leading '#', uppercase all columns
    pheno_df.columns = [c.lstrip("#").upper() for c in pheno_df.columns]
    cov_df.columns   = [c.lstrip("#").upper() for c in cov_df.columns]

    # Merge on IID
    merged = pheno_df.merge(cov_df, on="IID", how="inner", suffixes=("", "_cov"))
    merged = merged.reset_index(drop=True)

    # Identify phenotype column (everything in pheno that isn't IID/FID)
    disease_col = next(c for c in pheno_df.columns if c not in ("IID", "FID"))

    # Covariate columns: SEX, AGE, PC1-PC10
    pc_cols  = [f"PC{i}" for i in range(1, 11)]
    cov_cols = ["SEX", "AGE"] + pc_cols

    # Build rows explicitly so FID is guaranteed present
    rows = []
    for _, r in merged.iterrows():
        row = {"FID": "0", "IID": r["IID"], "PATID": "0", "MATID": "0", "SEX": r["SEX"] if "SEX" in merged.columns else "NA"}
        row[disease_col] = r[disease_col]
        for col in [c for c in cov_cols if c != "SEX"]:
            row[col] = r[col] if col in merged.columns else "NA"
        rows.append(row)

    out_df = pd.DataFrame(rows,
                          columns=["FID", "IID", "PATID", "MATID", "SEX",
                                   disease_col] + [c for c in cov_cols if c != "SEX"])
    out_df.to_csv(rv_file, sep="\t", index=False)
    print(f"  RVtests combined file → {rv_file}  "
          f"({len(out_df)} samples, cols: {list(out_df.columns)})")
    return rv_file

def run_rvtests(chnum, vcf_gz, covariate_file, out_dir, chr_genes=None):
    chr_tag   = f"chr{chnum}"
    chrom_dir = os.path.join(out_dir, chr_tag)
    out_base  = os.path.join(chrom_dir, f"{chr_tag}_rvtest")

    # Strip 'chr' prefix from VCF chromosome names if needed
    stripped_vcf_gz = out_base + "_nochr.vcf.gz"
    stripped_vcf    = out_base + "_nochr.vcf"

    with gzip.open(vcf_gz, "rt") as fin, open(stripped_vcf, "w") as fout:
        for line in fin:
            if line.startswith("#"):
                fout.write(line)
            else:
                parts = line.split("\t")
                parts[0] = re.sub(r"^chr", "", parts[0])
                fout.write("\t".join(parts))

    run(["bgzip", "-f", stripped_vcf])
    run(["tabix", "-p", "vcf", stripped_vcf_gz])

    # Build RVtests-format pheno and covariate files
    rv_file = make_rvtests_files(covariate_file, out_base)

    # Build gene value: comma-separated list of genes on this chromosome
    gene_arg = ",".join(sorted(chr_genes)) if chr_genes else None
    if gene_arg:
        print(f"[{chr_tag}] gene list: {len(chr_genes)} genes")

    # RVtests burden test
    rvtest_cmd = [
        RVTEST_BIN,
        "noweb",
        "hide-covar",
        "out", out_base,
        "kernel", "skat,skato",
        "inVcf", stripped_vcf_gz,
        "pheno", rv_file,
        "pheno-name", PHENO_NAME,
        "geneFile", REF_FLAT,
        "covar", rv_file,
        "covar-name", RVTEST_COVARS
    ]
    if gene_arg:
        rvtest_cmd += ["gene", gene_arg]

    run(rvtest_cmd)
    print(f"[{chr_tag}] RVtests done → {out_base}")
    return out_base


# Run RVtests – pass chr_genes from chrom_results for faster refFlat lookup
rvtest_prefixes = {}
for chnum, (vcf_gz, _, _) in extraction_results.items():
    _, _, chr_genes = chrom_results.get(chnum, (None, None, None))
    try:
        rvtest_prefixes[chnum] = run_rvtests(
            chnum, vcf_gz, COVARIATE_BASE, OUT_DIR, chr_genes=chr_genes
        )
    except Exception as e:
        print(f"chr{chnum} RVtests ERROR: {e}")


## Detailed ANNOVAR annotation (for final output table)

Run on the filtered variant set using full protocol:
`refGene, clinvar_20250721, dbnsfp47a, avsnp151, gnomad41_genome, gnomad41_exome`

In [ ]:
ANNOVAR_PROTOCOL   = "refGene,clinvar_20250721,dbnsfp47a,avsnp151,gnomad41_genome"
ANNOVAR_OPERATIONS = "g,f,f,f,f"

def annotate_detailed(chnum, snpeff_vcf, passed_df):
    chr_tag   = f"chr{chnum}"
    chrom_dir = os.path.join(OUT_DIR, chr_tag)

    # Write a minimal VCF containing only the passing variant positions
    filtered_vcf_gz = os.path.join(chrom_dir, f"{chr_tag}_filtered.vcf.gz")
    annovar_prefix  = os.path.join(chrom_dir, f"{chr_tag}_annovar_detail")

    run(["perl", ANNOVAR_PL, filtered_vcf_gz, ANNOVAR_DB,
         "-buildver", "hg38",
         "-out", annovar_prefix,
         "-remove",
         "-protocol", ANNOVAR_PROTOCOL,
         "-operation", ANNOVAR_OPERATIONS,
         "-nastring", ".",
         "-vcfinput",
         "thread", "1"])

    out_txt = f"{annovar_prefix}.hg38_multianno.txt"
    print(f"[{chr_tag}] Detailed ANNOVAR done → {out_txt}")
    return out_txt


detailed_annovar = {}
for chnum, (sv, at, cg) in chrom_results.items():
    if chnum not in all_passed or all_passed[chnum].empty:
        continue
    try:
        detailed_annovar[chnum] = annotate_detailed(chnum, sv, all_passed[chnum])
    except Exception as e:
        print(f"chr{chnum} detailed ANNOVAR ERROR: {e}")

## Parse snpEff VCF annotations

In [ ]:
def parse_snpeff_info(info_str):
    result = {"GeneID": ".", "GeneTranscript": ".", "Exon": ".",
              "cDNA": ".", "Protein": "."}
    ann_match = re.search(r'ANN=([^;]+)', info_str)
    if not ann_match:
        return result
    first_ann = ann_match.group(1).split(",")[0]
    fields = first_ann.split("|")
    if len(fields) >= 11:
        result["GeneID"]         = fields[3]   # GeneName
        result["GeneTranscript"] = fields[6]   # FeatureID (transcript)
        result["Exon"]           = fields[8]   # Rank (exon number)
        result["cDNA"]           = fields[9]   # HGVS.c
        result["Protein"]        = fields[10]  # HGVS.p
    return result


def parse_snpeff_vcf(snpeff_vcf):
    records = []
    with open(snpeff_vcf) as f:
        for line in f:
            if line.startswith("#"):
                continue
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 8:
                continue
            chrom, pos, varid, ref, alt, qual, filt, info = parts[:8]
            ann = parse_snpeff_info(info)
            records.append({
                "Chr": chrom, "Pos": pos, "rsID": varid,
                "Ref": ref,   "Alt": alt,
                **ann
            })
    return pd.DataFrame(records)

## Build final variant annotation table

In [ ]:
#Column mappings from ANNOVAR dbnsfp47a 
DBNSFP_COLS = {
    "CADD_phred":          "CADD",
    "SIFT_pred":           "SIFT",
    "Polyphen2_HDIV_pred": "Polyphen2",
    "AlphaMissense_pred":  "AlphaMissense",
    "PrimateAI_pred":      "PrimateAI",
    "GERP++_RS":           "GERP++",
}

MERGE_KEYS = ["Chr", "Pos", "Ref", "Alt"]

ANNOVAR_KEEP = [
    "CADD", "SIFT", "Polyphen2", "AlphaMissense", "PrimateAI", "GERP++",
    "gnomAD 4.1", "ClinVar",
    "avsnp151",
    "Func.refGene", "ExonicFunc.refGene", "Gene.refGene",
    "AAChange.refGene",
]

# 3-letter to 1-letter amino acid map (used for snpEff HGVS.p which is 3-letter)
AA3_TO_1 = {
    'Ala':'A','Arg':'R','Asn':'N','Asp':'D','Cys':'C',
    'Glu':'E','Gln':'Q','Gly':'G','His':'H','Ile':'I',
    'Leu':'L','Lys':'K','Met':'M','Phe':'F','Pro':'P',
    'Ser':'S','Thr':'T','Trp':'W','Tyr':'Y','Val':'V',
    'Ter':'*','Sec':'U','Pyl':'O',
}
_AA3_RE = re.compile(r'(?:Ala|Arg|Asn|Asp|Cys|Glu|Gln|Gly|His|Ile|Leu|Lys|Met|Phe|Pro|Ser|Thr|Trp|Tyr|Val|Ter|Sec|Pyl)')

def shorten_protein(p):
    """Convert 3-letter amino-acid HGVS.p to 1-letter (e.g. p.Lys41Arg → p.K41R)."""
    if p is None or (isinstance(p, float) and pd.isna(p)):
        return p
    s = str(p)
    if s in (".", "", "nan"):
        return s
    return _AA3_RE.sub(lambda m: AA3_TO_1.get(m.group(0), m.group(0)), s)


def parse_annovar_aachange(val):
    """
    ANNOVAR AAChange.refGene: GENE:NM_TRANSCRIPT:EXON:cDNA:PROTEIN,GENE:NM_...
    Keep first isoform. ANNOVAR's protein is already 1-letter (e.g. p.V35I).
    """
    empty = {"AnnovarTranscript": ".", "AnnovarExon": ".",
             "AnnovarcDNA": ".", "AnnovarProtein": ".", "FullTranscript": "."}
    if val is None or (isinstance(val, float) and pd.isna(val)) or str(val) in (".", "", "UNKNOWN"):
        return empty
    full = str(val)
    first = full.split(",")[0]
    parts = first.split(":")
    return {
        "AnnovarTranscript": parts[1] if len(parts) > 1 else ".",
        "AnnovarExon":       parts[2] if len(parts) > 2 else ".",
        "AnnovarcDNA":       parts[3] if len(parts) > 3 else ".",
        "AnnovarProtein":    parts[4] if len(parts) > 4 else ".",
        "FullTranscript":    full,
    }


def build_annotation_table():
    frames = []
    for chnum in sorted(all_passed.keys()):
        if all_passed[chnum].empty:
            continue
        chr_tag   = f"chr{chnum}"
        sv        = chrom_results[chnum][0]
        at_detail = detailed_annovar.get(chnum)

        #snpEff (cross-check; HGVS.p is 3-letter so we shorten it) 
        snpeff_df = parse_snpeff_vcf(sv)
        for col in MERGE_KEYS:
            snpeff_df[col] = snpeff_df[col].astype(str)
        snpeff_df["Chr"] = snpeff_df["Chr"].str.replace(r"^chr", "", regex=True)
        snpeff_df = snpeff_df.rename(columns={
            "GeneTranscript": "SnpEffTranscript",
            "Exon":           "SnpEffExon",
            "cDNA":           "SnpEffcDNA",
            "Protein":        "SnpEffProtein",
            "GeneID":         "SnpEffGene",
        })
        if "SnpEffProtein" in snpeff_df.columns:
            snpeff_df["SnpEffProteinShort"] = snpeff_df["SnpEffProtein"].apply(shorten_protein)

        #ANNOVAR detailed 
        if at_detail and os.path.exists(at_detail):
            av = pd.read_csv(at_detail, sep="\t", low_memory=False)
            av.columns = [c.strip() for c in av.columns]
            av = av.rename(columns={"Start": "Pos"})

            rename_map = {}
            for k, v in DBNSFP_COLS.items():
                matches = [c for c in av.columns if k in c]
                if matches:
                    rename_map[matches[0]] = v
            av = av.rename(columns=rename_map)

            gnomad_cols = [c for c in av.columns
                           if "gnomad41" in c.lower() and "AF" in c]
            if gnomad_cols:
                av["gnomAD 4.1"] = (
                    av[gnomad_cols].replace(".", np.nan)
                                   .apply(pd.to_numeric, errors="coerce")
                                   .min(axis=1)
                )

            clinvar_col = next((c for c in av.columns
                                if "CLNSIG" in c.upper()), None)
            if clinvar_col:
                av["ClinVar"] = av[clinvar_col]

            avsnp_col = next((c for c in av.columns
                              if "avsnp" in c.lower()), None)
            if avsnp_col and avsnp_col != "avsnp151":
                av = av.rename(columns={avsnp_col: "avsnp151"})

            keep = MERGE_KEYS + [c for c in ANNOVAR_KEEP if c in av.columns]
            av = av[keep].copy()
            for col in MERGE_KEYS:
                av[col] = av[col].astype(str)
            av["Chr"] = av["Chr"].str.replace(r"^chr", "", regex=True)

            if "AAChange.refGene" in av.columns:
                parsed = av["AAChange.refGene"].apply(parse_annovar_aachange).apply(pd.Series)
                av = pd.concat([av.drop(columns=["AAChange.refGene"]), parsed], axis=1)
            else:
                for c in ["AnnovarTranscript","AnnovarExon","AnnovarcDNA","AnnovarProtein","FullTranscript"]:
                    av[c] = "."

            merged = snpeff_df.merge(av, on=MERGE_KEYS, how="left")
        else:
            merged = snpeff_df.copy()
            for c in ["AnnovarTranscript","AnnovarExon","AnnovarcDNA","AnnovarProtein","FullTranscript"]:
                merged[c] = "."

        #Func/ExonicFunc/Gene fallback from all_passed 
        pf = all_passed[chnum].copy()
        pf["Chr"] = pf["Chr"].astype(str).str.replace(r"^chr", "", regex=True)
        pf["Pos"] = pf["Start"].astype(str)
        pf["Ref"] = pf["Ref"].astype(str)
        pf["Alt"] = pf["Alt"].astype(str)
        func_col   = next((c for c in pf.columns if "Func.refGene"       in c), None)
        exfunc_col = next((c for c in pf.columns if "ExonicFunc.refGene" in c), None)
        gene_col   = next((c for c in pf.columns if "Gene.refGene"       in c), None)
        extra = [c for c in [func_col, exfunc_col, gene_col]
                 if c and c not in merged.columns]
        if extra:
            merged = merged.merge(pf[MERGE_KEYS + extra],
                                  on=MERGE_KEYS, how="left")

        suffix_cols = [c for c in merged.columns
                       if c.endswith("_x") or c.endswith("_y")]
        merged = merged.drop(columns=suffix_cols)

        frames.append(merged)

    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


annotation_table = build_annotation_table()
print(f"Annotation table: {len(annotation_table)} rows, {len(annotation_table.columns)} columns")
print(f"Columns: {list(annotation_table.columns)}")

## Load association results and merge with annotations

In [ ]:
import glob
from io import StringIO

def glob_first(directory, pattern, exclude_suffix=".adjusted"):
    full_pattern = os.path.join(directory, pattern)
    matches = [f for f in sorted(glob.glob(full_pattern))
               if not f.endswith(exclude_suffix)]
    if not matches:
        print(f"    glob found nothing: {full_pattern}")
    return matches[0] if matches else None


def get_chrom_dir(chnum):
    return os.path.join(OUT_DIR, f"chr{chnum}")


def _read_adjusted(path_main, id_col):
    if path_main is None:
        return None
    adj = path_main + ".adjusted"
    if not os.path.exists(adj):
        d = os.path.dirname(path_main)
        base = os.path.basename(path_main)
        cand = [f for f in glob.glob(os.path.join(d, base + "*"))
                if f.endswith(".adjusted")]
        if not cand:
            return None
        adj = cand[0]
    try:
        adf = pd.read_csv(adj, sep=r"\s+", engine="python")
    except Exception as e:
        print(f"    could not read {adj}: {e}")
        return None
    adf.columns = [c.lstrip("#") for c in adf.columns]
    id_in_adj = None
    for cand in (id_col, "ID", "SNP"):
        if cand in adf.columns:
            id_in_adj = cand
            break
    bonf_col = next((c for c in adf.columns if c.upper() == "BONF"), None)
    if id_in_adj is None or bonf_col is None:
        return None
    out = adf[[id_in_adj, bonf_col]].rename(
        columns={id_in_adj: id_col, bonf_col: "BONF"})
    print(f"    BONF merged from {os.path.basename(adj)} ({len(out)} rows)")
    return out


def _attach_bonf(df, path_main, id_col):
    if df is None or df.empty:
        return df
    bonf = _read_adjusted(path_main, id_col)
    if bonf is None:
        return df
    if "BONF" in df.columns:
        df = df.drop(columns=["BONF"])
    return df.merge(bonf, on=id_col, how="left")


def load_plink_fisher(chnum):
    p = glob_first(get_chrom_dir(chnum), "*fisher*.assoc.fisher")
    if not p:
        return None
    print(f"  fisher: {os.path.basename(p)}")
    df = pd.read_csv(p, sep=r"\s+", engine="python")
    df = _attach_bonf(df, p, "SNP")
    return df.rename(columns={"SNP": "ID"}) if "SNP" in df.columns else df


def load_plink_chisq(chnum):
    d = get_chrom_dir(chnum)
    full_pattern = os.path.join(d, "*chi*.assoc")
    matches = [f for f in sorted(glob.glob(full_pattern))
               if not f.endswith(".adjusted")
               and not any(f.endswith(x) for x in
                           [".bed", ".bim", ".fam", ".log", ".nosex"])]
    if not matches:
        print(f"    glob found nothing: {full_pattern}")
        return None
    p = matches[0]
    print(f"  chisq: {os.path.basename(p)}")
    df = pd.read_csv(p, sep=r"\s+", engine="python")
    df = _attach_bonf(df, p, "SNP")
    return df.rename(columns={"SNP": "ID"}) if "SNP" in df.columns else df


def load_glm(chnum, suffix):
    d = get_chrom_dir(chnum)
    pattern = os.path.join(d, "*.glm.logistic.hybrid")
    candidates = [f for f in sorted(glob.glob(pattern))
                  if not f.endswith(".adjusted")]

    if suffix == "nocovar":
        matches = [f for f in candidates if "nocovar" in os.path.basename(f)]
    elif suffix == "covar":
        matches = [f for f in candidates
                   if "covar" in os.path.basename(f)
                   and "nocovar" not in os.path.basename(f)]
    else:
        matches = [f for f in candidates if suffix in os.path.basename(f)]

    if not matches:
        print(f"    no GLM '{suffix}' file in {d}")
        return None
    p = matches[0]
    print(f"  glm-{suffix}: {os.path.basename(p)}")

    with open(p) as fh:
        lines = [l for l in fh if not l.startswith("##")]
    if not lines:
        return None
    lines[0] = lines[0].lstrip("#")
    df = pd.read_csv(StringIO("".join(lines)), sep="\t")
    df = _attach_bonf(df, p, "ID")
    return df


def load_rvtest(chnum, test_type="skat"):
    pref = rvtest_prefixes.get(chnum)
    if not pref:
        return None
    suffix_map = {"skat": "Skat.assoc", "skato": "SkatO.assoc"}
    filepath = f"{pref}.{suffix_map.get(test_type.lower(), test_type + '.assoc')}"
    if not os.path.exists(filepath):
        print(f"    [chr{chnum}] RVtest file not found: {filepath}")
        return None
    return pd.read_csv(filepath, sep="\t")


print("Loader functions defined.")
if extraction_results:
    test_chnum = next(iter(extraction_results))
    d = get_chrom_dir(test_chnum)
    print(f"Sanity check chr{test_chnum} → {d}")
    print(f"  glm files: {[os.path.basename(f) for f in sorted(glob.glob(os.path.join(d, '*.glm.logistic.hybrid'))) if not f.endswith('.adjusted')]}")


In [ ]:
#Collect all association results 
fisher_frames  = []
chisq_frames   = []
glm_nc_frames  = []
glm_cov_frames = []
skat_frames    = []
skato_frames   = []

for chnum in extraction_results:
    for loader, store in [
        (load_plink_fisher, fisher_frames),
        (load_plink_chisq,  chisq_frames),
        (lambda c: load_glm(c, "nocovar"), glm_nc_frames),
        (lambda c: load_glm(c, "covar"),   glm_cov_frames),
        (lambda c: load_rvtest(c, "skat"),  skat_frames),
        (lambda c: load_rvtest(c, "skato"), skato_frames),
    ]:
        result = loader(chnum)
        if result is not None:
            store.append(result)


def concat_clean(frames):
    """Concatenate frames, drop chnum if present."""
    if not frames:
        return pd.DataFrame()
    df = pd.concat(frames, ignore_index=True)
    if "chnum" in df.columns:
        df = df.drop(columns=["chnum"])
    return df


fisher_df  = concat_clean(fisher_frames)
chisq_df   = concat_clean(chisq_frames)
glm_nc_df  = concat_clean(glm_nc_frames)
glm_cov_df = concat_clean(glm_cov_frames)
skat_df    = concat_clean(skat_frames)
skato_df   = concat_clean(skato_frames)

print(f"Fisher: {len(fisher_df)} | ChiSq: {len(chisq_df)} | "
      f"GLM no-covar: {len(glm_nc_df)} | GLM covar: {len(glm_cov_df)} | "
      f"SKAT: {len(skat_df)} | SKAT-O: {len(skato_df)}")

#Replace rsID with avsnp151 (already in annotation_table from ANNOVAR) 
def patch_rsid_from_avsnp(annotation_table):
    if "avsnp151" not in annotation_table.columns:
        print("  avsnp151 column not found – rsID unchanged.")
        return annotation_table
    at = annotation_table.copy()
    if "rsID" not in at.columns:
        at["rsID"] = "."
    mask = at["avsnp151"].notna() & (at["avsnp151"] != ".")
    at.loc[mask, "rsID"] = at.loc[mask, "avsnp151"]
    at = at.drop(columns=["avsnp151"])   # folded into rsID
    print(f"  rsID patched with avsnp151: {mask.sum()} / {len(at)} variants")
    return at


annotation_table = patch_rsid_from_avsnp(annotation_table)


## Build the final merged output table

Columns: `GeneID  GeneTranscript  Exon  cDNA  Protein  rsID  Func  ExonicFunc  ClinVar  gnomAD 4.1  CADD  SIFT  Polyphen2  AlphaMissense  PrimateAI  GERP++  P-value (Fisher)  P Bonferroni (Fisher)  OR (Fisher)  SE (Fisher)  FullTranscript  OBSCT`

In [ ]:
def _norm_chr(s):
    return s.astype(str).str.replace(r"^chr", "", regex=True)


def parse_plink_id(df, id_col):
    df = df.copy()
    parts = df[id_col].astype(str).str.split(":", expand=True)
    df["Chr"] = _norm_chr(parts[0])
    df["Pos"] = parts[1] if parts.shape[1] > 1 else pd.NA
    df["Ref"] = parts[2] if parts.shape[1] > 2 else pd.NA
    df["Alt"] = parts[3] if parts.shape[1] > 3 else pd.NA
    return df


#Determine total cases and controls from the pheno file 
def _count_cases_controls():
    pheno_path = f"{COVARIATE_BASE}.pheno"
    if not os.path.exists(pheno_path):
        print(f"  WARNING: pheno file not found at {pheno_path}")
        return None, None
    df = pd.read_csv(pheno_path, sep=r"\s+", engine="python")
    df.columns = [c.lstrip("#") for c in df.columns]
    pheno_col = next((c for c in df.columns
                      if c.upper() in ("DISEASE","PHENO","PHENOTYPE")), None)
    if pheno_col is None:
        # take the last non-id column
        ids = {"FID","IID"}
        non_id = [c for c in df.columns if c not in ids]
        pheno_col = non_id[-1] if non_id else None
    if pheno_col is None:
        return None, None
    vals = pd.to_numeric(df[pheno_col], errors="coerce")
    n_case = int((vals == 2).sum())
    n_ctrl = int((vals == 1).sum())
    print(f"  Pheno file → {n_case} cases, {n_ctrl} controls (from '{pheno_col}')")
    return n_case, n_ctrl


N_CASES, N_CTRLS = _count_cases_controls()


#Per-test stat extractors 
def _extract_plink19_assoc(df, label):
    if df is None or df.empty:
        return pd.DataFrame()
    id_col = "ID" if "ID" in df.columns else ("SNP" if "SNP" in df.columns else None)
    if id_col is None:
        return pd.DataFrame()

    out = parse_plink_id(df, id_col)
    rename = {}
    if "P"    in out.columns: rename["P"]    = f"P-value ({label})"
    if "OR"   in out.columns: rename["OR"]   = f"OR ({label})"
    if "SE"   in out.columns: rename["SE"]   = f"SE ({label})"
    if "BONF" in out.columns: rename["BONF"] = f"P Bonferroni ({label})"
    out = out.rename(columns=rename)

    # Frequencies → counts
    if "F_A" in out.columns:
        out[f"Freq cases ({label})"] = pd.to_numeric(out["F_A"], errors="coerce")
        if N_CASES is not None:
            out[f"Count cases ({label})"] = (
                out[f"Freq cases ({label})"] * 2 * N_CASES
            ).round().astype("Int64")
    if "F_U" in out.columns:
        out[f"Freq controls ({label})"] = pd.to_numeric(out["F_U"], errors="coerce")
        if N_CTRLS is not None:
            out[f"Count controls ({label})"] = (
                out[f"Freq controls ({label})"] * 2 * N_CTRLS
            ).round().astype("Int64")

    keep = ["Chr","Pos","Ref","Alt"] + [v for v in rename.values()] + [
        c for c in [
            f"Freq cases ({label})", f"Count cases ({label})",
            f"Freq controls ({label})", f"Count controls ({label})",
        ] if c in out.columns
    ]
    out = out[[c for c in keep if c in out.columns]].drop_duplicates(["Chr","Pos","Ref","Alt"])
    for c in ["Chr","Pos","Ref","Alt"]:
        out[c] = out[c].astype(str)
    return out


def _extract_plink2_glm(df, label):
    if df is None or df.empty:
        return pd.DataFrame()
    id_col = "ID" if "ID" in df.columns else ("SNP" if "SNP" in df.columns else None)
    if id_col is None:
        return pd.DataFrame()

    out = parse_plink_id(df, id_col)

    # Pick exactly one OR-style column (prefer ORBETA, else OR)
    or_src = "ORBETA" if "ORBETA" in out.columns else ("OR" if "OR" in out.columns else None)
    se_src = "LOG(OR)_SE" if "LOG(OR)_SE" in out.columns else ("SE" if "SE" in out.columns else None)

    rename = {}
    if "P"    in out.columns: rename["P"]    = f"P-value ({label})"
    if or_src:                rename[or_src] = f"OR ({label})"
    if se_src:                rename[se_src] = f"SE ({label})"
    if "BONF" in out.columns: rename["BONF"] = f"P Bonferroni ({label})"
    out = out.rename(columns=rename)

    # Case / control counts and frequencies (already in plink2 output)
    count_freq_map = {
        "A1_CASE_CT":  f"Count cases ({label})",
        "A1_CTRL_CT":  f"Count controls ({label})",
        "A1_CASE_FREQ":f"Freq cases ({label})",
        "A1_CTRL_FREQ":f"Freq controls ({label})",
        "A1_FREQ":     f"Freq all ({label})",
        "A1_CT":       f"Count all ({label})",
        "OBS_CT":      f"N tested ({label})",
    }
    for src, dst in count_freq_map.items():
        if src in out.columns:
            out = out.rename(columns={src: dst})

    # Fallback: if counts missing but freqs+totals present, derive
    if N_CASES is not None and f"Count cases ({label})" not in out.columns and f"Freq cases ({label})" in out.columns:
        out[f"Count cases ({label})"] = (
            pd.to_numeric(out[f"Freq cases ({label})"], errors="coerce") * 2 * N_CASES
        ).round().astype("Int64")
    if N_CTRLS is not None and f"Count controls ({label})" not in out.columns and f"Freq controls ({label})" in out.columns:
        out[f"Count controls ({label})"] = (
            pd.to_numeric(out[f"Freq controls ({label})"], errors="coerce") * 2 * N_CTRLS
        ).round().astype("Int64")

    keep_cols = ["Chr","Pos","Ref","Alt"] + list(rename.values()) + [
        f"Count cases ({label})", f"Count controls ({label})",
        f"Freq cases ({label})",  f"Freq controls ({label})",
        f"Count all ({label})",   f"Freq all ({label})",
        f"N tested ({label})",
    ]
    seen = set()
    keep = [c for c in keep_cols if c in out.columns and not (c in seen or seen.add(c))]
    out = out[keep].drop_duplicates(["Chr","Pos","Ref","Alt"])
    for c in ["Chr","Pos","Ref","Alt"]:
        out[c] = out[c].astype(str)
    return out


#Annotation base 
def _annotated_base(annotation_table):
    base = annotation_table.copy()
    if "Chr" in base.columns:
        base["Chr"] = _norm_chr(base["Chr"])
    for col in ["Pos","Ref","Alt"]:
        if col in base.columns:
            base[col] = base[col].astype(str)

    if "Func.refGene"       in base.columns: base = base.rename(columns={"Func.refGene":"Func"})
    if "ExonicFunc.refGene" in base.columns: base = base.rename(columns={"ExonicFunc.refGene":"ExonicFunc"})
    if "Gene.refGene"       in base.columns: base = base.rename(columns={"Gene.refGene":"Gene"})

    if "avsnp151" in base.columns:
        base["rsID"] = base["avsnp151"].where(
            base["avsnp151"].notna() & (base["avsnp151"] != "."), other=".")
        base = base.drop(columns=["avsnp151"], errors="ignore")
    elif "rsID" not in base.columns:
        base["rsID"] = "."

    rename_to_primary = {
        "AnnovarTranscript": "GeneTranscript",
        "AnnovarExon":       "Exon",
        "AnnovarcDNA":       "cDNA",
        "AnnovarProtein":    "Protein",
    }
    for src, dst in rename_to_primary.items():
        if src in base.columns:
            base = base.rename(columns={src: dst})

    return base


def _build_user_columns(label):
    return [
        "Gene", "ID", "GeneTranscript", "Exon", "cDNA", "Protein",
        "rsID", "Func", "ExonicFunc", "ClinVar", "gnomAD 4.1",
        "CADD", "SIFT", "Polyphen2", "AlphaMissense", "PrimateAI", "GERP++",
        f"Count cases ({label})",    f"Freq cases ({label})",
        f"Count controls ({label})", f"Freq controls ({label})",
        f"Count all ({label})",      f"Freq all ({label})",
        f"N tested ({label})",
        f"P-value ({label})",        f"P Bonferroni ({label})",
        f"OR ({label})",             f"SE ({label})",
        # snpEff cross-check
        "SnpEffTranscript", "SnpEffExon", "SnpEffcDNA",
        "SnpEffProtein",    "SnpEffProteinShort",
        "FullTranscript",
    ]


def build_assoc_sheet(annotation_table, assoc_df, label, kind="plink2"):
    base = _annotated_base(annotation_table)
    if "ID" not in base.columns:
        base["ID"] = (base["Chr"].astype(str) + ":" + base["Pos"].astype(str)
                      + ":" + base["Ref"].astype(str) + ":" + base["Alt"].astype(str))

    if kind == "plink19":
        stats = _extract_plink19_assoc(assoc_df, label)
    else:
        stats = _extract_plink2_glm(assoc_df, label)

    print(f"  [{label}] base={len(base)}  stats={len(stats)}")
    if not stats.empty and not base.empty:
        base_keys = set(zip(base["Chr"], base["Pos"], base["Ref"], base["Alt"]))
        stat_keys = set(zip(stats["Chr"], stats["Pos"], stats["Ref"], stats["Alt"]))
        overlap = len(base_keys & stat_keys)
        print(f"  [{label}] key overlap: {overlap}")
        if overlap == 0 and base_keys and stat_keys:
            print(f"    sample base key: {next(iter(base_keys))}")
            print(f"    sample stat key: {next(iter(stat_keys))}")

        merged = base.merge(stats, on=["Chr","Pos","Ref","Alt"],
                            how="inner", suffixes=("", "_drop"))
        merged = merged.drop(columns=[c for c in merged.columns if c.endswith("_drop")])
    else:
        merged = base.iloc[0:0].copy()

    cols = [c for c in _build_user_columns(label) if c in merged.columns]
    out = merged[cols].copy()
    print(f"  [{label}] sheet: {len(out)} rows, {len(out.columns)} cols")
    return out


fisher_sheet  = build_assoc_sheet(annotation_table, fisher_df,  "Fisher",    kind="plink19")
chisq_sheet   = build_assoc_sheet(annotation_table, chisq_df,   "ChiSq",     kind="plink19")
glm_nc_sheet  = build_assoc_sheet(annotation_table, glm_nc_df,  "GLM",       kind="plink2")
glm_cov_sheet = build_assoc_sheet(annotation_table, glm_cov_df, "GLM+covar", kind="plink2")

final_table = fisher_sheet.copy()
print(f"\nfinal_table (= fisher_sheet): {len(final_table)} rows, {len(final_table.columns)} cols")
final_table.head(3)


#### Sheet 1: Variant functional annotation counts per gene

In [ ]:
FUNC_CATEGORIES = [
    "Exonic", "Nonsynonymous SNV", "Synonymous SNV", "Startloss",
    "Stopgain", "Stoploss", "Frameshift deletion", "Frameshift insertion",
    "Frameshift substitution", "Nonframeshift deletion",
    "Nonframeshift insertion", "Nonframeshift substitution",
    "Intronic", "Splicing", "UTR3", "UTR5", "Downstream", "Upstream",
    "ncRNA exonic", "ncRNA intronic", "ncRNA splicing"
]

EXTRA_CATEGORIES = [
    "Pathogenic", "Likely pathogenic", "VUS",
    "CADD>=12", "CADD>=20",
]


def _classify_clinvar(val):
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return None
    s = str(val).lower()
    if s in (".", "", "nan"):
        return None
    if "likely_pathogenic" in s or "likely pathogenic" in s:
        return "Likely pathogenic"
    if "pathogenic" in s and "conflicting" not in s:
        return "Pathogenic"
    if "uncertain_significance" in s or "uncertain significance" in s or s == "vus":
        return "VUS"
    return None


def _to_float(v):
    try:
        f = float(v)
        if pd.isna(f):
            return None
        return f
    except (TypeError, ValueError):
        return None


def categorize_func(func, exfunc):
    func = str(func).lower()
    exfunc = str(exfunc)
    if "splicing" in func and "ncrna" not in func:    return "Splicing"
    if func == "intronic":           return "Intronic"
    if "utr3" in func:               return "UTR3"
    if "utr5" in func:               return "UTR5"
    if "downstream" in func:         return "Downstream"
    if "upstream" in func:           return "Upstream"
    if "ncrna" in func and "exon"     in func: return "ncRNA exonic"
    if "ncrna" in func and "intron"   in func: return "ncRNA intronic"
    if "ncrna" in func and "splicing" in func: return "ncRNA splicing"
    if "exonic" in func:
        exf = exfunc.lower()
        if "nonsynonymous"           in exf: return "Nonsynonymous SNV"
        if "synonymous"              in exf: return "Synonymous SNV"
        if "stopgain"                in exf: return "Stopgain"
        if "stoploss"                in exf: return "Stoploss"
        if "startloss"               in exf: return "Startloss"
        if "frameshift deletion"     in exf: return "Frameshift deletion"
        if "frameshift insertion"    in exf: return "Frameshift insertion"
        if "frameshift substitution" in exf: return "Frameshift substitution"
        if "nonframeshift deletion"  in exf: return "Nonframeshift deletion"
        if "nonframeshift insertion" in exf: return "Nonframeshift insertion"
        if "nonframeshift substitution" in exf: return "Nonframeshift substitution"
        return "Exonic"
    return "Exonic"


def build_gene_count_sheet(annotation_table):
    base_frames = []
    for chnum, pf in all_passed.items():
        if pf.empty:
            continue
        gene_col   = next((c for c in pf.columns if "Gene.refGene"       in c), None)
        func_col   = next((c for c in pf.columns if "Func.refGene"       in c), None)
        exfunc_col = next((c for c in pf.columns if "ExonicFunc.refGene" in c), None)
        if not gene_col:
            continue
        tmp = pf[["Chr","Start","Ref","Alt", gene_col,
                  func_col or gene_col, exfunc_col or gene_col]].copy()
        tmp.columns = ["Chr","Pos","Ref","Alt","Gene","Func","ExonicFunc"]
        # CRITICAL: strip 'chr' prefix so the merge with annotation_table works
        tmp["Chr"] = tmp["Chr"].astype(str).str.replace(r"^chr","",regex=True)
        for c in ["Pos","Ref","Alt"]:
            tmp[c] = tmp[c].astype(str)
        base_frames.append(tmp)

    if not base_frames:
        return pd.DataFrame(columns=["Gene"] + FUNC_CATEGORIES + EXTRA_CATEGORIES)

    df = pd.concat(base_frames, ignore_index=True)
    df["Category"] = df.apply(lambda r: categorize_func(r["Func"], r["ExonicFunc"]), axis=1)
    df["Gene"] = df["Gene"].astype(str).str.split(r"[;,]").str[0]

    # Functional category pivot
    pivot = (
        df.groupby(["Gene","Category"])
          .size()
          .unstack(fill_value=0)
          .reindex(columns=FUNC_CATEGORIES, fill_value=0)
          .reset_index()
    )
    pivot["Exonic"] = pivot[FUNC_CATEGORIES[1:]].sum(axis=1)

    # Bring in CADD + ClinVar from annotation_table
    at = annotation_table.copy()
    if "Chr" in at.columns:
        at["Chr"] = at["Chr"].astype(str).str.replace(r"^chr","",regex=True)
    for c in ["Pos","Ref","Alt"]:
        if c in at.columns:
            at[c] = at[c].astype(str)
    cadd_col   = "CADD"    if "CADD"    in at.columns else None
    clin_col   = "ClinVar" if "ClinVar" in at.columns else None
    keep = ["Chr","Pos","Ref","Alt"] + [c for c in (cadd_col, clin_col) if c]
    at_slim = at[[c for c in keep if c in at.columns]].drop_duplicates(["Chr","Pos","Ref","Alt"])

    # Diagnostic
    df_keys = set(zip(df["Chr"], df["Pos"], df["Ref"], df["Alt"]))
    at_keys = set(zip(at_slim["Chr"], at_slim["Pos"], at_slim["Ref"], at_slim["Alt"]))
    print(f"  CADD/ClinVar enrichment: {len(df_keys & at_keys)} of {len(df_keys)} variants matched")
    if not (df_keys & at_keys) and df_keys and at_keys:
        print(f"    sample passed key:     {next(iter(df_keys))}")
        print(f"    sample annotation key: {next(iter(at_keys))}")

    enriched = df.merge(at_slim, on=["Chr","Pos","Ref","Alt"], how="left")

    enriched["_clin"] = enriched[clin_col].apply(_classify_clinvar) if clin_col else None
    enriched["_cadd"] = enriched[cadd_col].apply(_to_float)         if cadd_col else None

    agg = enriched.groupby("Gene").apply(
        lambda g: pd.Series({
            "Pathogenic":        int((g["_clin"] == "Pathogenic").sum())        if clin_col else 0,
            "Likely pathogenic": int((g["_clin"] == "Likely pathogenic").sum()) if clin_col else 0,
            "VUS":               int((g["_clin"] == "VUS").sum())               if clin_col else 0,
            "CADD>=12":          int(g["_cadd"].dropna().ge(12).sum())          if cadd_col else 0,
            "CADD>=20":          int(g["_cadd"].dropna().ge(20).sum())          if cadd_col else 0,
        })
    ).reset_index()

    pivot = pivot.merge(agg, on="Gene", how="left").fillna(0)
    for c in EXTRA_CATEGORIES:
        pivot[c] = pivot[c].astype(int)

    return pivot


gene_count_sheet = build_gene_count_sheet(annotation_table)
print(f"Gene count sheet: {len(gene_count_sheet)} genes, {len(gene_count_sheet.columns)} cols")
gene_count_sheet.head(5)


## Aggregate all results into Excel workbook

In [ ]:
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter
import openpyxl

OUTPUT_XLSX = os.path.join(OUT_DIR, "results_summary_n44_WGS.xlsx")

def _build_variant_catalog():
    rows = []
    for chnum, pf in all_passed.items():
        if pf.empty:
            continue
        func_col   = next((c for c in pf.columns if "Func.refGene"       in c), None)
        exfunc_col = next((c for c in pf.columns if "ExonicFunc.refGene" in c), None)
        for _, r in pf.iterrows():
            try:
                pos = int(r["Start"])
            except (TypeError, ValueError):
                continue
            chrom = str(r["Chr"]).replace("chr", "")
            cat = categorize_func(
                r[func_col]   if func_col   else "",
                r[exfunc_col] if exfunc_col else "")
            rows.append({"Chr": chrom, "Pos": pos, "Category": cat})
    return pd.DataFrame(rows)


VARIANT_CATALOG = _build_variant_catalog()
print(f"Variant catalog for SKAT range counts: {len(VARIANT_CATALOG)} variants")


_RANGE_RE = re.compile(r"(?:chr)?([0-9XYMT]+):(\d+)-(\d+)")

def _counts_for_range(range_str):
    out = {c: 0 for c in FUNC_CATEGORIES}
    out["TestedVariants"] = 0
    if not range_str or VARIANT_CATALOG.empty:
        return out
    for piece in str(range_str).split(","):
        m = _RANGE_RE.search(piece.strip())
        if not m:
            continue
        chrom, start, end = m.group(1), int(m.group(2)), int(m.group(3))
        sub = VARIANT_CATALOG[
            (VARIANT_CATALOG["Chr"] == chrom) &
            (VARIANT_CATALOG["Pos"] >= start) &
            (VARIANT_CATALOG["Pos"] <= end)
        ]
        out["TestedVariants"] += len(sub)
        for cat, n in sub["Category"].value_counts().items():
            if cat in out:
                out[cat] += int(n)
    return out


def annotate_burden_with_func(burden_df, label=""):
    if burden_df is None or burden_df.empty:
        print(f"  [{label}] empty — nothing to annotate")
        return burden_df
    print(f"  [{label}] columns: {list(burden_df.columns)}")
    range_col = next((c for c in burden_df.columns
                      if c.lower() in ("range","range_","gene_range")), None)
    if not range_col:
        print(f"  [{label}] no Range column — skipping per-range func counts")
        return burden_df
    print(f"  [{label}] using Range column: {range_col!r}")
    counts = burden_df[range_col].apply(_counts_for_range).apply(pd.Series)
    keep_cols = [c for c in counts.columns
                 if c == "TestedVariants" or counts[c].sum() > 0]
    counts = counts[keep_cols]
    print(f"  [{label}] added per-range count columns: {list(counts.columns)}")
    return pd.concat([burden_df, counts], axis=1)


print("\n- SKAT annotation -")
skat_sheet  = annotate_burden_with_func(skat_df,  "SKAT")
print("\n- SKAT-O annotation -")
skato_sheet = annotate_burden_with_func(skato_df, "SKAT-O")


def auto_format_sheet(ws):
    header_fill = PatternFill("solid", fgColor="4472C4")
    for cell in ws[1]:
        cell.font      = Font(bold=True, color="FFFFFF")
        cell.fill      = header_fill
        cell.alignment = Alignment(horizontal="center", wrap_text=True)
    ws.freeze_panes = "A2"
    for col_cells in ws.columns:
        max_len = max((len(str(c.value or "")) for c in col_cells), default=10)
        ws.column_dimensions[get_column_letter(col_cells[0].column)].width = min(max_len + 2, 40)


def df_to_sheet(wb, sheet_name, df):
    if df is None or df.empty:
        ws = wb.create_sheet(sheet_name)
        ws.append(["No data available"])
        return
    ws = wb.create_sheet(sheet_name)
    ws.append(list(df.columns))
    for _, row in df.iterrows():
        ws.append([None if pd.isna(v) else v for v in row])
    auto_format_sheet(ws)


wb = openpyxl.Workbook()
wb.remove(wb.active)

df_to_sheet(wb, "1_Gene_Variant_Counts", gene_count_sheet)
df_to_sheet(wb, "2_SKAT",                skat_sheet)
df_to_sheet(wb, "3_SKAT-O",              skato_sheet)
df_to_sheet(wb, "4_Fisher_Assoc",        fisher_sheet)
df_to_sheet(wb, "5_ChiSq_Assoc",         chisq_sheet)
df_to_sheet(wb, "6_GLM_NoCovar",         glm_nc_sheet)
df_to_sheet(wb, "7_GLM_WithCovar",       glm_cov_sheet)
df_to_sheet(wb, "8_Variant_Annotations", final_table)

wb.save(OUTPUT_XLSX)
print(f"\n✓ Workbook saved → {OUTPUT_XLSX}")
print("Sheets:", [ws.title for ws in wb.worksheets])
